# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hassaan-Raza/FlyRank-Intership/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one content item (content_hash_id), with metrics aggregated across
a full calendar month. Time window: March 2026 (2026-03-01 to 2026-03-31),
a mid-panel month, not the sealed final month (2026-06) reserved for testing.

In [1]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

grain_check = con.sql(f"""
    SELECT content_hash_id, COUNT(*) as row_count
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id
    LIMIT 10
""").df()
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id  row_count
0  content_7a105f548d9c6916         31
1  content_a3ea9792f793ec72         31
2  content_36c36abc7650d7af         31
3  content_a7da352b73b02668         31
4  content_f39be42b42a4e8f6         31
5  content_1855a661b4d36130         31
6  content_5d412fba6e1a2582         31
7  content_1f380a642aed423b         31
8  content_22c063002b7c1caf         31
9  content_aafb2ab7e5fc80d0         31


## 2. Fields: feature / label / context / excluded

Features: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, and a
derived days_with_impressions count (observable search/analytics signals,
safe to use).

Label/proxy: a decline flag built from gsc_impressions dropping between the
first and second half of the window.

Context: content_hash_id, client_hash_id (for joins and grouping only, not
features themselves).

Excluded: any FlyRank product-computed score (health_score, priority_score,
action_type), and any row where ga4_data_available is FALSE. Also excluding
AI-platform columns (sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini,
ai_copilot, ai_claude, ai_other), those belong to a separate freestyle lane.
Excluded because product scores would let the model copy an existing rule
instead of finding real signal, and rows without GA4 tracking yet would
misrepresent "no traffic" as real zero traffic.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# Query 1: row count and date span for this slice
span_check = con.sql(f"""
    SELECT COUNT(*) as total_rows,
           MIN(report_date) as earliest_date,
           MAX(report_date) as latest_date
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(span_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows earliest_date latest_date
0     9841378    2026-03-01  2026-03-31


In [4]:
# Query 2: availability check, filtered with IS TRUE
availability_check = con.sql(f"""
    SELECT COUNT(*) as total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) as available_rows
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(availability_check)
print(f"Survival rate: {availability_check['available_rows'][0] / availability_check['total_rows'][0]:.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  available_rows
0     9841378          413966
Survival rate: 4.2%


In [5]:
# Query 3: missing values check on key feature columns
missing_check = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(*) - COUNT(gsc_impressions) as missing_impressions,
        COUNT(*) - COUNT(gsc_avg_position) as missing_position
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(missing_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  missing_impressions  missing_position
0     9841378                    0           6230317


In [6]:
# Five-feature frame for this lane
features = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) as total_impressions,
        SUM(gsc_clicks) as total_clicks,
        AVG(gsc_avg_position) as avg_position,
        SUM(ga4_sessions) as total_sessions,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) as days_with_impressions
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE ga4_data_available IS TRUE
    GROUP BY content_hash_id
""").df()
print(features.head(10))
print(f"\nShape: {features.shape}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id  total_impressions  total_clicks  avg_position  \
0  content_5e120e972f11f833                0.0           0.0           NaN   
1  content_4ab81290aec524dd                0.0           0.0           NaN   
2  content_b1f61fc81b28b2d4              458.0           2.0      4.418032   
3  content_e25ea7297a1dffd3             3943.0          23.0      4.392897   
4  content_aba6e5270431d8ef                0.0           0.0           NaN   
5  content_3c286ded8bd68120             2180.0          15.0      8.439390   
6  content_9d17d30b63eaa640                0.0           0.0           NaN   
7  content_b2108e8fe3360fa6              503.0           8.0      5.531459   
8  content_0535f4407e4320df                0.0           0.0           NaN   
9  content_ff867882e604fa96               24.0           0.0      2.850000   

   total_sessions  days_with_impressions  
0             3.0                      0  
1             1.0                      0  
2           

## 4. Data limits

This slice only covers March 2026 for clients whose GA4 tracking had already
started by that point. Only 4.2% of rows have ga4_data_available IS TRUE
(413,966 of 9,841,378), so any GA4-dependent feature (sessions, engagement)
is only usable on a small fraction of this month's data, worth treating as
a hard constraint, not a detail. gsc_avg_position is also missing on the
majority of rows (6,230,317 of 9,841,378, about 63%), likely because
position is only recorded when the page actually ranked for a query that
day, this directly affects the avg_position feature's reliability. GSC-only
rows (no GA4 yet) can still misrepresent "no traffic" as real zero traffic
if this filter is skipped. This window also can't tell us anything about
seasonality beyond one month, or about clients who joined after March.

In [7]:
print(availability_check)
print(missing_check)

   total_rows  available_rows
0     9841378          413966
   total_rows  missing_impressions  missing_position
0     9841378                    0           6230317


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.